# WGBS Preprocessing — Step 1: Build Beta Matrices

**Goal:** Read per-sample Bismark ROI TSV files from 4 project directories and save clean CpG × sample beta-value matrices to `data/processed/`, ready for the ML pipeline.

| Dataset key | Source | Comparison | n (approx) |
|---|---|---|---|
| `stress` | nhip_cumulus | Control vs Stressed | ~70 |
| `wildfire` | macaque_nasal_HongJi | Control vs Exposed | ~22 |
| `obesity_hippocampus` | fromBen_MacaqueObese_brain / Hippocampus_OvC | Control vs Obese | ~13 |
| `obesity_hypothalamus` | fromBen_MacaqueObese_brain / Hypothalamus_OvC | Control vs Obese | ~13 |
| `obesity_prefrontalcortex` | fromBen_MacaqueObese_brain / PrefrontalCortex_OvC | Control vs Obese | ~13 |
| `cfdna_GD90` | fromBen_MacaqueObese_cfDna / GD90_OvC | Control vs Obese | ~12 |
| `cfdna_GD120` | fromBen_MacaqueObese_cfDna / GD120_OvC | Control vs Obese | ~12 |
| `cfdna_GD150` | fromBen_MacaqueObese_cfDna / GD150_OvC | Control vs Obese | ~12 |

**Outputs per dataset** (saved to `data/processed/`):
- `{dataset}_cpgi_methylation.csv` — CpGs × samples, beta values, CpG island window
- `{dataset}_genebody_methylation.csv` — CpGs × samples, beta values, gene body window
- `{dataset}_labels.csv` — sample_id, label (1=case, 0=control)

## Cell 1 — Imports

**What to remember:**
- `pathlib.Path` is the modern way to handle file paths in Python. It's cleaner than string concatenation and works on all operating systems.
- `glob.glob()` finds files matching a pattern (like `*.tsv.gz`). Think of it as a file search.
- `numpy` (as `np`) handles numerical arrays and math operations efficiently.
- `pandas` (as `pd`) handles tabular data — you'll use it constantly in bioinformatics.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

print('Libraries loaded.')

## Cell 2 — Configuration

**What to remember:** Always put ALL paths and parameters at the top of your notebook/script in one place. Never hardcode them inside functions. This is called the **config block** pattern — when something changes (a path, a threshold), you change it once here, not in ten different places.

**Key concept — `Path(__file__).resolve().parent`:** In a script, `__file__` is the script's own path. In a notebook, we use `Path.cwd()` (current working directory) instead. Since this notebook lives in `notebooks/`, we go one level up (`..`) to reach the project root.

**`MAX_MISSING = 0.20`** means: if a CpG has no coverage in more than 20% of samples, drop it. With n=13 samples, that allows up to ~2 samples to be missing at a CpG.

In [ ]:
# ── Project root (one level up from this notebook) ──────────────────
PROJECT_ROOT = Path.cwd().parent
# Path.cwd() = current working directory = where this notebook runs from
# .parent    = go one folder up

# ── Output directory ────────────────────────────────────────────────
OUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
# parents=True  → create all folders in the path if they don't exist
# exist_ok=True → don't raise an error if the folder already exists

# ── Parent directory containing all four dataset project folders ─────
PARENT = Path('/project/nhip_macaque')
# Change this if your base directory is different

# ── Genomic region of interest ───────────────────────────────────────
CHROM        = 'chr10'
REGION_START = 2432261
REGION_END   = 2443263

# Sub-feature windows (used to split the full matrix into two outputs)
CPG_START      = 2433175
CPG_END        = 2433562
GENEBODY_START = REGION_START + 1250  # = 2433511
GENEBODY_END   = REGION_END   - 1000  # = 2442263

# ── Filtering parameters ─────────────────────────────────────────────
MIN_COV     = 1     # minimum read coverage per CpG per sample
MAX_MISSING = 0.20  # drop CpGs missing in >20% of samples

print(f'Output directory : {OUT_DIR}')
print(f'Region           : {CHROM}:{REGION_START:,}-{REGION_END:,}')
print(f'CpG island window: {CPG_START:,}-{CPG_END:,}')
print(f'Gene body window : {GENEBODY_START:,}-{GENEBODY_END:,}')

## Cell 3 — Dataset Configuration

**What to remember:** This is a **dictionary of dictionaries** — a very common Python pattern for configuration. The outer key (e.g. `'stress'`) becomes the output file prefix. Each inner dict holds everything the processing function needs to know about that dataset.

**`parse_mode`** tells the parser how to extract the sample ID from the filename — each dataset has different naming conventions:
- `'roi1kb'` → stress + wildfire: `HJD036_merged_....roi1kb.tsv.gz` → `HJD036`
- `'brain'`  → obesity regions: `47383-Hippocampus_....roi.tsv.gz` → `47383-Hippocampus`
- `'cfdna'`  → cfDNA: `100224688.roi.tsv.gz` → `100224688`

**`meta_filter`** is used for cfDNA only — the metadata file covers all gestational days, so we filter it to get only the samples belonging to a specific GD timepoint.

In [ ]:
DATASETS = {

    # ── Stress (NHIP preconception) ──────────────────────────────────
    'stress': {
        'roi_dir'     : PARENT / 'nhip_cumulus' / 'roi_tsvfiles',
        'metadata'    : PARENT / 'nhip_cumulus' / 'sample_info.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*.roi1kb.tsv.gz',
        'parse_mode'  : 'roi1kb',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Stressed',
        'meta_filter' : None,           # no filtering needed
    },

    # ── Wildfire smoke (nasal epithelial) ────────────────────────────
    'wildfire': {
        'roi_dir'     : PARENT / 'macaque_nasal_HongJi' / 'roi_tsvfiles',
        'metadata'    : PARENT / 'macaque_nasal_HongJi' / 'sample_info.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*.roi1kb.tsv.gz',
        'parse_mode'  : 'roi1kb',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Exposed',
        'meta_filter' : None,
    },

    # ── Obesity — Hippocampus ─────────────────────────────────────────
    'obesity_hippocampus': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'Hippocampus_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
        # Brain region is already encoded in the Name (e.g. 47383-Hippocampus)
        # so filenames and metadata match naturally without extra filtering
    },

    # ── Obesity — Hypothalamus ────────────────────────────────────────
    'obesity_hypothalamus': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'Hypothalamus_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
    },

    # ── Obesity — Prefrontal Cortex ───────────────────────────────────
    'obesity_prefrontalcortex': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'PrefrontalCortex_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
    },

    # ── cfDNA — Gestational Day 90 ────────────────────────────────────
    'cfdna_GD90': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD90_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD90_OvC'),
        # meta_filter = (column_name, value) — filters metadata to this GD only
        # The cfDNA metadata covers all GDs in one file, so we must filter
    },

    # ── cfDNA — Gestational Day 120 ───────────────────────────────────
    'cfdna_GD120': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD120_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD120_OvC'),
    },

    # ── cfDNA — Gestational Day 150 ───────────────────────────────────
    'cfdna_GD150': {
        'roi_dir'     : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD150_OvC',
        'metadata'    : PARENT / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roi.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD150_OvC'),
    },
}

print(f'{len(DATASETS)} datasets configured:')
for name in DATASETS:
    print(f'  {name}')

## Cell 4 — Helper Functions

### `parse_sample_name()` — Why three modes?
In real bioinformatics, files from different pipelines/labs have different naming conventions. Rather than three separate scripts, we handle all cases in one function using a `mode` argument. This is a pattern you'll write constantly.

### `load_sample()` — Key concepts:
- **`pd.read_csv(..., header=None, names=cols)`** — when a file has no column headers (Bismark output), you assign them yourself
- **`compression='gzip'`** — pandas reads `.gz` files directly, no need to unzip
- **`np.where(condition, value_if_true, value_if_false)`** — vectorised conditional, faster and safer than a for-loop
- **`.copy()`** — always call this after filtering a DataFrame to avoid `SettingWithCopyWarning`

### `build_matrix()` — The core pattern:
Each sample becomes a `pd.Series` with CpG positions as the index. When you pass a dictionary of Series to `pd.DataFrame()`, pandas **automatically aligns them on their index** — CpGs present in one sample but not another become `NaN`. This one line replaces what would otherwise be a complex merge loop.

In [ ]:
# ── 1. Sample name parser ────────────────────────────────────────────

def parse_sample_name(fpath: str, mode: str) -> str:
    """
    Extract sample ID from filename. Three modes for three naming conventions.

    roi1kb : HJD036_merged_....roi1kb.tsv.gz     → HJD036
    brain  : 47383-Hippocampus_ZR3377_....roi.tsv.gz → 47383-Hippocampus
    cfdna  : 100224688.roi.tsv.gz                → 100224688
    """
    base = os.path.basename(fpath)  # strip directory path, keep filename only

    if mode == 'roi1kb':
        base = base.replace('.roi1kb.tsv.gz', '')
        return base.split('_', 1)[0]          # everything before first '_'

    elif mode == 'brain':
        base = base.replace('.roi.tsv.gz', '')
        return base.split('_', 1)[0]          # e.g. '47383-Hippocampus'

    elif mode == 'cfdna':
        return base.replace('.roi.tsv.gz', '') # just strip suffix, ID is the whole name

    else:
        raise ValueError(f'Unknown parse_mode: {mode}')


# ── 2. Load one sample ───────────────────────────────────────────────

def load_sample(fpath: str) -> pd.DataFrame:
    """
    Read one Bismark ROI TSV file. Returns a DataFrame with columns:
      start : CpG genomic position (used as the row identifier)
      beta  : methylation level 0-1 (= meth_reads / total_reads)

    Returns empty DataFrame if no usable CpGs in the region.
    """
    cols = ['chrom', 'start', 'end', 'pct', 'meth', 'unmeth']

    df = pd.read_csv(
        fpath,
        sep='\t',            # Bismark files are tab-separated
        header=None,         # no column names in the file
        names=cols,          # we assign names ourselves
        compression='gzip'   # read .gz directly — no need to unzip first
    )

    if df.empty:
        return df

    # Filter to correct chromosome
    df = df[df['chrom'] == CHROM].copy()
    if df.empty:
        return df

    # Filter to region of interest using start position
    df = df[
        (df['start'] >= REGION_START) &
        (df['start'] <= REGION_END)
    ].copy()
    if df.empty:
        return df

    # Compute total coverage and beta value
    df['cov']  = df['meth'].astype(float) + df['unmeth'].astype(float)
    df['beta'] = np.where(
        df['cov'] > 0,                             # if covered:
        df['meth'].astype(float) / df['cov'],      #   beta = meth / total
        np.nan                                      # else: missing
    )
    # np.where is safer than plain division — avoids ZeroDivisionError

    # Apply minimum coverage filter
    df = df[df['cov'] >= MIN_COV].copy()
    df = df.dropna(subset=['beta'])

    return df[['start', 'beta']].reset_index(drop=True)


# ── 3. Build CpG × samples matrix ───────────────────────────────────

def build_matrix(files, name_to_group, ctrl_label, case_label, parse_mode):
    """
    Load all sample files and combine into one CpG × samples DataFrame.

    Core trick:
      Each sample → pd.Series(index=CpG_positions, values=beta)
      pd.DataFrame(dict_of_series) aligns all series on their index automatically
      Missing positions become NaN
    """
    sample_series = {}   # {sample_name: pd.Series(beta values)}
    labels        = {}   # {sample_name: 0 or 1}

    for fpath in files:
        name  = parse_sample_name(fpath, parse_mode)
        group = name_to_group.get(name)   # .get() returns None if key missing

        if group is None:
            print(f'  [SKIP] {name} — not in metadata')
            continue
        if group not in (ctrl_label, case_label):
            print(f'  [SKIP] {name} — group "{group}" not in comparison')
            continue

        df = load_sample(fpath)
        if df.empty:
            print(f'  [SKIP] {name} — no data in region')
            continue

        # Convert DataFrame row to Series: index=position, value=beta
        s = df.set_index('start')['beta']
        s.name = name
        sample_series[name] = s
        labels[name] = 1 if group == case_label else 0
        # Convention: case (exposed/disease) = 1, control = 0

    if not sample_series:
        raise RuntimeError('No samples loaded. Check paths and metadata Name column.')

    # This single line does all the alignment across samples:
    # pd.DataFrame({name1: series1, name2: series2, ...})
    # rows = union of all CpG positions, columns = sample names
    beta_matrix = pd.DataFrame(sample_series)
    beta_matrix.index.name = 'CpG_start'

    print(f'  Loaded: {beta_matrix.shape[1]} samples × {beta_matrix.shape[0]} CpGs (before filtering)')
    return beta_matrix, labels


# ── 4. Filter CpGs with too much missing data ────────────────────────

def filter_missing(beta_matrix):
    """
    Drop CpGs missing in more than MAX_MISSING fraction of samples.

    isna()       → True/False for each cell
    .sum(axis=1) → count NaN per ROW (per CpG) — axis=1 means 'across columns'
    / n_samples  → fraction missing
    """
    n        = beta_matrix.shape[1]                    # number of samples
    frac_na  = beta_matrix.isna().sum(axis=1) / n     # fraction missing per CpG
    kept     = beta_matrix[frac_na <= MAX_MISSING].copy()
    dropped  = beta_matrix.shape[0] - kept.shape[0]
    print(f'  Missing filter: dropped {dropped} CpGs → {kept.shape[0]} retained')
    return kept


# ── 5. Subset to a genomic window ────────────────────────────────────

def subset_window(beta_matrix, start, end, name):
    """
    Slice the matrix to CpGs within [start, end].
    The matrix index holds CpG start positions, so we filter on the index.
    """
    mask   = (beta_matrix.index >= start) & (beta_matrix.index <= end)
    subset = beta_matrix.loc[mask].copy()
    print(f'  {name:20s}: {subset.shape[0]} CpGs')
    return subset


print('Helper functions defined.')

## Cell 5 — Main Processing Loop

**What to remember:** `for name, cfg in DATASETS.items()` is how you loop over a dictionary getting both the key and value simultaneously. `.items()` returns (key, value) pairs — you'll use this constantly.

**`glob.glob()`** finds all files matching a pattern. The `str()` call is needed because glob expects a string, not a Path object.

**`dict(zip(meta['Name'], meta['Group']))`** — this is a very common bioinformatics one-liner: `zip()` pairs two lists together, `dict()` converts those pairs into a fast lookup table. Result: `{'HJD036': 'Control', 'HJD037': 'Exposed', ...}`

In [ ]:
summary_rows = []   # collect per-dataset results for the final summary table

for ds_name, cfg in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")

    # ── Find sample files ────────────────────────────────────────────
    files = sorted(glob.glob(str(cfg['roi_dir'] / cfg['file_pattern'])))
    # glob.glob returns a list of all matching file paths
    # str() converts Path to string because glob expects strings
    # sorted() ensures consistent order across runs

    if not files:
        print(f'  [ERROR] No files found in {cfg["roi_dir"]} — skipping.')
        continue
    print(f'  Files found: {len(files)}')

    # ── Load metadata ────────────────────────────────────────────────
    meta = pd.read_csv(cfg['metadata'], sep=cfg['meta_sep'])
    meta['Name']  = meta['Name'].astype(str)
    meta['Group'] = meta['Group'].astype(str)

    # Apply metadata filter for cfDNA (filter to the specific GD timepoint)
    if cfg['meta_filter'] is not None:
        col, val = cfg['meta_filter']    # unpack tuple: ('Folder', 'GD120_OvC')
        meta = meta[meta[col] == val].copy()
        print(f'  Metadata filtered to {col}={val}: {len(meta)} samples')

    # Build name→group lookup dictionary
    name_to_group = dict(zip(meta['Name'], meta['Group']))
    # zip() pairs Name and Group columns together
    # dict() converts those pairs into a fast lookup: {'SampleA': 'Control', ...}

    ctrl = cfg['ctrl_label']
    case = cfg['case_label']
    n_ctrl = sum(1 for g in name_to_group.values() if g == ctrl)
    n_case = sum(1 for g in name_to_group.values() if g == case)
    print(f'  Metadata: {n_ctrl} {ctrl} / {n_case} {case}')

    # ── Build beta matrix ────────────────────────────────────────────
    try:
        beta_matrix, labels = build_matrix(
            files, name_to_group, ctrl, case, cfg['parse_mode']
        )
    except RuntimeError as e:
        print(f'  [ERROR] {e}')
        continue

    # ── Filter missing CpGs ──────────────────────────────────────────
    beta_matrix = filter_missing(beta_matrix)

    # ── Split into sub-feature windows ───────────────────────────────
    cpgi_mat = subset_window(beta_matrix, CPG_START,      CPG_END,      'CpG island')
    gene_mat = subset_window(beta_matrix, GENEBODY_START, GENEBODY_END, 'Gene body')

    # ── Save output files ────────────────────────────────────────────
    cpgi_path  = OUT_DIR / f'{ds_name}_cpgi_methylation.csv'
    gene_path  = OUT_DIR / f'{ds_name}_genebody_methylation.csv'
    label_path = OUT_DIR / f'{ds_name}_labels.csv'

    cpgi_mat.to_csv(cpgi_path)
    gene_mat.to_csv(gene_path)
    # to_csv() writes the index (CpG positions) as the first column
    # and column names (sample IDs) as the header — exactly what ml_pipeline.py expects

    labels_df = pd.DataFrame(
        list(labels.items()),
        columns=['sample_id', 'label']
    )
    labels_df.to_csv(label_path, index=False)
    # index=False → don't write row numbers (0,1,2,...) to file

    print(f'  Saved: {cpgi_path.name}')
    print(f'         {gene_path.name}')
    print(f'         {label_path.name}')

    summary_rows.append({
        'dataset'      : ds_name,
        'n_samples'    : len(labels),
        'n_ctrl'       : sum(1 for v in labels.values() if v == 0),
        'n_case'       : sum(1 for v in labels.values() if v == 1),
        'cpgi_cpgs'    : cpgi_mat.shape[0],
        'genebody_cpgs': gene_mat.shape[0],
    })

print(f"\n{'='*60}")
print('All datasets processed.')
print(f"{'='*60}")

## Cell 6 — Summary Table

Always inspect your processed data before running ML. This table tells you:
- How many samples loaded successfully per dataset
- Whether your class balance is reasonable
- How many CpGs survived filtering per window

If a dataset has 0 samples loaded, that almost always means the sample names in the filenames don't match the `Name` column in the metadata — the most common preprocessing bug.

In [ ]:
summary = pd.DataFrame(summary_rows)
summary = summary.set_index('dataset')
print('\n── Preprocessing Summary ──')
print(summary.to_string())

# Save summary to results/tables/
tables_dir = PROJECT_ROOT / 'results' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(tables_dir / 'preprocessing_summary.csv')
print(f'\nSummary saved to results/tables/preprocessing_summary.csv')

## Cell 7 — Spot Check

Always look at a slice of your actual data after processing. This confirms:
- Beta values are between 0 and 1 (if you see values > 1, something went wrong)
- Sample IDs look correct
- CpG positions are within your expected window

**`df.head()`** shows the first 5 rows. **`df.describe()`** gives min/max/mean — essential for sanity checking.

In [ ]:
# Spot-check one dataset — change 'stress' to any dataset key
check_ds = 'stress'

cpgi_check = pd.read_csv(OUT_DIR / f'{check_ds}_cpgi_methylation.csv', index_col=0)
print(f'=== {check_ds} — CpG island matrix ===')
print(f'Shape: {cpgi_check.shape[0]} CpGs × {cpgi_check.shape[1]} samples')
print(f'\nFirst 5 rows:')
display(cpgi_check.head())

print(f'\nBeta value range (should be 0-1):')
print(f'  Min: {cpgi_check.values.min():.4f}')
print(f'  Max: {cpgi_check.values.max():.4f}')
print(f'  Mean: {cpgi_check.values.mean():.4f}')

labels_check = pd.read_csv(OUT_DIR / f'{check_ds}_labels.csv')
print(f'\nLabels:')
display(labels_check)